In [ ]:
#라이브러리 임포트
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

#데이터 불러오기
titanic_df = pd.read_csv('./titanic_train.csv')

#결측치 처리 함수 정의
def fillna(df):
    #Age 열의 결측치를 평균 나이로 채움
    df['Age'] = df['Age'].fillna(df['Age'].mean())
    #Cabin 열 결측치는 'N'으로 대체
    df['Cabin'] = df['Cabin'].fillna('N')
    #Embarked 열 결측치는 'N'으로 대체
    df['Embarked'] = df['Embarked'].fillna('N')
    #Fare 열 결측치는 0으로 대체
    df['Fare'] = df['Fare'].fillna(0)
    return df

#불필요한 열 제거 함수 정의
def drop_features(df):
    #모델 학습에 불필요한 PassengerId, Name, Ticket 제거
    df = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)
    return df

#범주형 데이터 인코딩 및 형식 변환 함수 정의
def format_features(df):
    #Cabin은 첫 글자만 추출하여 객실 등급으로 활용
    df['Cabin'] = df['Cabin'].str[:1]
    #LabelEncoder를 적용할 범주형 특징 리스트
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        #각 컬럼에 대해 라벨 인코딩 수행
        df[feature] = le.fit_transform(df[feature])
    return df

#전처리 함수 통합
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

#타이타닉 데이터 전처리 실행
titanic_df = transform_features(titanic_df)

#독립 변수와 종속 변수 분리
X = titanic_df.drop('Survived', axis=1)  # 입력 데이터 (특징들)
y = titanic_df['Survived']               # 타깃 데이터 (생존 여부)

#학습용과 테스트용 데이터 분리 (80% 학습, 20% 테스트)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#모델 생성 - 랜덤 포레스트 분류기 사용
model = RandomForestClassifier(random_state=42)

#모델 학습
model.fit(X_train, y_train)

#테스트 데이터로 예측 수행
y_pred = model.predict(X_test)

#예측 결과 평가 - 정확도 출력
accuracy = accuracy_score(y_test, y_pred)
print(f"모델 정확도: {accuracy:.4f}")